# Safe Formula Interpreter — Demo

The Simple Steps formula bar uses Python **syntax** but is **not** run through `eval`.
Every formula is parsed to an AST and walked by a small, auditable interpreter that
only knows how to do a closed set of things:

- Look up step variables (`step1`, `step2`, …)
- Read columns: `step1.url` or `step1["url"]`
- Call functions registered via `@simple_step`
- Use literals (numbers, strings, bools, lists, dicts)
- Nest those calls — the Excel-like part

Everything else (imports, method calls, attribute writes, dunder access, lambdas,
comprehensions, …) is rejected at validation time.

This notebook walks through the public API: `parse`, `validate`, `describe`,
and `run_formula`.


In [1]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
from SIMPLE_STEPS.decorators import simple_step, OPERATION_REGISTRY
from SIMPLE_STEPS.safe_formula import (
    parse, validate, describe, run_formula, FormulaError,
)
print("SIMPLE_STEPS loaded. Registered ops so far:", len(OPERATION_REGISTRY))



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py",

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "/Users/stuartsynakowski/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py",

AttributeError: _ARRAY_API not found

SIMPLE_STEPS loaded. Registered ops so far: 0


## 1. Define some operations

These are ordinary Python functions. `@simple_step` registers them so the
formula interpreter knows they're callable. Without registration, they can't
be called from a formula — that's the whole point.


In [2]:
@simple_step(id="demo_upper", category="Demo")
def demo_upper(text: str) -> str:
    """Uppercase a string."""
    return text.upper()

@simple_step(id="demo_word_count", category="Demo")
def demo_word_count(text: str) -> int:
    """Count words in a string."""
    return len(str(text).split())

@simple_step(id="demo_tag", category="Demo")
def demo_tag(text: str, prefix: str = "[", suffix: str = "]") -> str:
    """Wrap text with a prefix/suffix."""
    return f"{prefix}{text}{suffix}"

@simple_step(id="demo_add", category="Demo")
def demo_add(a: int, b: int) -> int:
    return a + b

[op for op in OPERATION_REGISTRY if op.startswith("demo_")]


['demo_upper', 'demo_word_count', 'demo_tag', 'demo_add']

## 2. A tiny dataset

`step1` will be a `DataFrame` of pretend video rows. The interpreter will
auto-wrap it in a `StepProxy` so that `step1.title` resolves to the column.


In [3]:
df = pd.DataFrame({
    "title": ["intro to pandas", "advanced sql", "hello world"],
    "views": [120, 540, 17],
})
steps = {"step1": df}
df


,title,views
0,intro to pandas,120
1,advanced sql,540
2,hello world,17


## 3. `describe()` — static analysis, no execution

`describe(formula)` is what the UI calls on every keystroke to power
red-squiggles and "missing argument" tooltips. It returns the top-level op,
every call node, and every step reference it can see.


In [4]:
describe("=demo_tag(text=demo_upper(text=step1.title), prefix='<<', suffix='>>')")


{'valid': True,
 'errors': [],
 'calls': [{'op': 'demo_tag',
   'kwargs': ['text', 'prefix', 'suffix'],
   'positional_count': 0},
  {'op': 'demo_upper', 'kwargs': ['text'], 'positional_count': 0}],
 'step_refs': ['step1.title'],
 'top_level_op': 'demo_tag'}

## 4. `validate()` — does this formula reference known things?

`validate` cross-checks every Name against the operation registry and the set
of available steps. Empty list = valid.


In [5]:
good = "=demo_upper(text=step1.title)"
bad_unknown_op = "=nuke_the_disk(path='/')"
bad_method     = "=step1.title.upper()"
bad_dunder     = "=demo_upper(text=step1.__class__)"

for f in [good, bad_unknown_op, bad_method, bad_dunder]:
    print(f)
    errs = validate(f, available_steps={"step1"})
    print("  ->", "OK" if not errs else errs)
    print()


=demo_upper(text=step1.title)
  -> OK

=nuke_the_disk(path='/')
  -> ["Unknown operation: 'nuke_the_disk'", "Unknown name: 'nuke_the_disk'"]

=step1.title.upper()
  -> ['Only direct calls to registered operations are allowed (no method calls, no computed callables).', "Attribute access is only allowed on step references (got '.upper' on non-step expression)."]

=demo_upper(text=step1.__class__)
  -> ["Private/dunder attribute access not allowed: '.__class__'"]



## 5. `run_formula()` — actually execute

`run_formula` parses, re-validates, then walks the tree. Calls dispatch through
the same auto-broadcasting wrapper used everywhere else in Simple Steps, so
passing a column reference (`step1.title`) automatically maps the function
row-wise.


In [6]:
# Pure literal call — no broadcasting, returns a scalar.
print(run_formula("=demo_add(a=2, b=40)", steps={}))


42


In [7]:
# Column-broadcast call — auto-maps over rows.
result = run_formula("=demo_upper(text=step1.title)", steps=steps)
result.df


,title,views,demo_upper_output
0,intro to pandas,120,INTRO TO PANDAS
1,advanced sql,540,ADVANCED SQL
2,hello world,17,HELLO WORLD


In [8]:
# Chaining via intermediate step variables — this is how the workflow
# runner composes formulas across steps.
step2 = run_formula("=demo_word_count(text=step1.title)", steps=steps)
steps_after = {"step1": df, "step2": step2.df}

step3 = run_formula("=demo_tag(text=step1.title, prefix='*', suffix='*')", steps=steps_after)
step3.df


,title,views,demo_tag_output
0,intro to pandas,120,*intro to pandas*
1,advanced sql,540,*advanced sql*
2,hello world,17,*hello world*


## 6. The security boundary

These are the kinds of strings that, under raw `eval`, would be catastrophic.
With `run_formula` they are rejected before any function is called.


In [9]:
attacks = [
    "=__import__('os').system('echo PWNED')",
    "=open('/etc/passwd').read()",
    "=step1.title.upper()",                  # method call — not allowed
    "=demo_upper(text=step1.__class__)",     # dunder access
    "=(lambda: 1)()",                        # lambdas blocked
    "=[x for x in range(10)]",               # comprehensions blocked
]
for f in attacks:
    try:
        run_formula(f, steps=steps)
        print("UNEXPECTED:", f)
    except FormulaError as e:
        first = str(e).splitlines()[0]
        print(f"BLOCKED  {f!r:55s}  -> {first}")


BLOCKED  "=__import__('os').system('echo PWNED')"                 -> Invalid formula:
BLOCKED  "=open('/etc/passwd').read()"                            -> Invalid formula:
BLOCKED  '=step1.title.upper()'                                   -> Invalid formula:
BLOCKED  '=demo_upper(text=step1.__class__)'                      -> Invalid formula:
BLOCKED  '=(lambda: 1)()'                                         -> Invalid formula:
BLOCKED  '=[x for x in range(10)]'                                -> Invalid formula:
